## Required libraries installation

In [1]:
%pip install accelerate scipy librosa -q
%pip install transformers torchaudio pandas datasets -q
%pip install soundfile -q
%pip install transformers==4.44.2 -q
%pip install tqdm -q

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## All imports and custom built modules and functions

In [2]:
# Normal imports
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import gc

# Torch imports
import torch
import torchaudio
import torchaudio.transforms as T
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import ASTFeatureExtractor


/home/warden/anaconda3/envs/pytorch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Device agnostic code

In [3]:
! nvidia-smi

Wed Oct 14 21:30:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.57.04              KMD Version: 610.57.04     CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   50C    P8              2W /  115W |       5MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
device = "cuda" if torch.cuda.is_available else "cpu"
device
# if it shows cpu download cuda torch audio

'cuda'

### Inspecting the data on some nonchalant shii

In [1]:
! python preprocessing_pipeline.py

      EcoAST TRAIN / VAL / TEST PIPELINE          
Indexing raw audio files for FSD50K mapping...
Collected total of 41817 raw audio records.

Standardizing audio files for split: 'TRAIN' (29271 samples)...
Processing train: 100%|█████████████████| 29271/29271 [04:26<00:00, 109.75it/s]
Exported 29271 rows to metadata: split_dataset/train_metadata.csv

Standardizing audio files for split: 'VAL' (6273 samples)...
Processing val: 100%|█████████████████████| 6273/6273 [00:56<00:00, 111.84it/s]
Exported 6273 rows to metadata: split_dataset/val_metadata.csv

Standardizing audio files for split: 'TEST' (6273 samples)...
Processing test: 100%|████████████████████| 6273/6273 [00:57<00:00, 109.81it/s]
Exported 6273 rows to metadata: split_dataset/test_metadata.csv

             PIPELINE EXPORT COMPLETE             


In [6]:
from preprocessing_pipeline import get_dataloaders

from training import AST_VARIANTS, train_ast_model, ForestThreatAST

NUM_CLASSES = 4  # chainsaw, gunshot, engine, background
BATCH_SIZE = 2
EPOCHS = 5

#Training AST 10x10

In [7]:
variant_name = "AST-AudioSet-10x10"
checkpoint_name = AST_VARIANTS[variant_name]["checkpoint"]

print(f"Loading DataLoaders for {variant_name}...")
train_loader_v1, val_loader_v1, test_loader_v1 = get_dataloaders(
    output_dir="split_dataset",
    batch_size=BATCH_SIZE,
    model_checkpoint=checkpoint_name,
    num_workers=8,
)

print(f"Initializing {variant_name}...")
model_v1 = ForestThreatAST(num_classes=NUM_CLASSES, variant_name=variant_name).to(
    device
)

print(f"Starting Training...")
model_v1 = train_ast_model(
    model=model_v1,
    train_loader=train_loader_v1,
    device=device,
    model_name=variant_name,
    num_epochs=EPOCHS,
    save_path=f"trained_models/ecoast_{variant_name}.pth",
)

Loading DataLoaders for AST-AudioSet-10x10...
Initializing AST-AudioSet-10x10...


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 7.65 GiB of which 23.19 MiB is free. Including non-PyTorch memory, this process has 7.62 GiB memory in use. Of the allocated memory 7.43 GiB is allocated by PyTorch, and 64.05 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# Training AST_SPEECH_COMMAND_V2

In [ ]:
variant_name = "AST-SpeechCommands-V2"
checkpoint_name = AST_VARIANTS[variant_name]["checkpoint"]

print(f"Loading DataLoaders for {variant_name}...")
train_loader_v2, val_loader_v2, test_loader_v2 = get_dataloaders(
    output_dir="split_dataset", batch_size=BATCH_SIZE, model_checkpoint=checkpoint_name
)

print(f"Initializing {variant_name}...")
model_v2 = ForestThreatAST(num_classes=NUM_CLASSES, variant_name=variant_name).to(
    device
)

print(f"Starting Training...")
model_v2 = train_ast_model(
    model=model_v2,
    train_loader=train_loader_v2,
    device=device,
    model_name=variant_name,
    num_epochs=EPOCHS,
    save_path=f"trained_models/ecoast_{variant_name}.pth",
)

# Training AST-Fast-Stride-16x16

In [ ]:
variant_name = "AST-FastStride-16x16"
checkpoint_name = AST_VARIANTS[variant_name]["checkpoint"]

print(f"Loading DataLoaders for {variant_name}...")
# Re-using the AudioSet checkpoint for feature extraction, but the AST config will stride differently
train_loader_v3, val_loader_v3, test_loader_v3 = get_dataloaders(
    output_dir="split_dataset", batch_size=BATCH_SIZE, model_checkpoint=checkpoint_name
)

print(f"Initializing {variant_name}...")
model_v3 = ForestThreatAST(num_classes=NUM_CLASSES, variant_name=variant_name).to(
    device
)

print(f"Starting Training...")
model_v3 = train_ast_model(
    model=model_v3,
    train_loader=train_loader_v3,
    device=device,
    model_name=variant_name,
    num_epochs=NUM_CLASSES,
    save_path=f"trained_models/ecoast_{variant_name}.pth",
)